# Neuronal Tracer — Step 1: Preprocessing & Tubularity

Unified GPU + CPU notebook. Device priority: **MPS → CUDA → CPU**.

## Outputs

| File | Contents |
|------|----------|
| `output/tubularity.npz` | `T_combined`, `scale_idx_m`, `radius_map`, `orient_field`, `sigmas`, `voxel_iso` |
| `output/stack_iso.tif` | Isotropic-rescaled fluorescence stack (input to step2) |
| `output/T_combined.tif` | Tubularity map for visual QC in Fiji/ImageJ |

## Pipeline

| Step | Description |
|------|-------------|
| 1 | Load raw TIFF; auto-detect XY voxel size from metadata |
| 2 | Visualisation helpers (MIP, slice browser) |
| 3 | Downsample → normalise → isotropic Z-rescale → sigma conversion |
| 4 | Multi-scale Hessian: Frangi + Meijering tubularity, orientation field |

In [ ]:
# Run once to install all dependencies, then restart the kernel before proceeding.
# !pip install tifffile scikit-image scipy numpy matplotlib ipywidgets torch

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 100,
    'figure.facecolor': '#111111',
    'axes.facecolor': '#111111',
    'axes.titlecolor': 'white',
    'text.color': 'white',
})

# ── GPU Detection ─────────────────────────────────────────────────────────────
# Priority: Apple Silicon MPS > NVIDIA CUDA > CPU fallback.
# If PyTorch is not installed, USE_GPU=False and the pipeline falls back to
# scikit-image (frangi / meijering) running on CPU. All other code is shared.
try:
    import torch
    import torch.nn.functional as F

    def _get_device():
        if torch.backends.mps.is_available():
            return torch.device('mps')
        if torch.cuda.is_available():
            return torch.device('cuda')
        return torch.device('cpu')

    DEVICE  = _get_device()
    USE_GPU = True
    print(f'PyTorch {torch.__version__} — device: {DEVICE}')
    if DEVICE.type == 'mps':
        print('  Apple Silicon GPU (Metal Performance Shaders)')
    elif DEVICE.type == 'cuda':
        print(f'  NVIDIA {torch.cuda.get_device_name(0)}')
        print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    else:
        print('  PyTorch installed but no GPU found — running on CPU via PyTorch')
        print('  Consider using the skimage CPU path (set USE_GPU=False manually) for simpler debugging.')

except ImportError:
    USE_GPU = False
    DEVICE  = None
    print('PyTorch not found — using scikit-image CPU path.')
    print('To enable GPU acceleration: pip install torch')

# scikit-image is always imported regardless of USE_GPU:
#   - CPU fallback path uses frangi() and meijering() directly
#   - Step 2 (skeletonization) uses morphology functions from skimage
from skimage.filters import frangi, meijering
print('scikit-image loaded (CPU fallback ready).')
print('All packages loaded successfully.')

---
## Configuration

All parameters are in **physical units (µm)** where applicable.
Sigma values are converted to voxels automatically after isotropic rescaling (Step 3d).

Edit this cell and re-run it; Steps 3d onward will pick up the new values.
Steps 1–3c only need re-running if `DATA_PATH`, `VOXEL_Z`, or `DOWNSAMPLE_*` change.

In [ ]:
# ── Input ─────────────────────────────────────────────────────────────────────
DATA_PATH = '/Users/lee/Tracer/data/sample_crop.tif'   # (Z, Y, X) axis order required
VOXEL_Z   = 1.0    # µm — Z slice thickness; set from microscope acquisition settings

# ── Working Resolution ─────────────────────────────────────────────────────────
# DOWNSAMPLE_XY = 2 halves each XY dimension (4× fewer voxels per slice).
# Use 1 for a final production run; 2–4 for quick iteration.
DOWNSAMPLE_XY = 2
DOWNSAMPLE_Z  = 1

# ── Tube Radius Range (µm) ─────────────────────────────────────────────────────
# Sets the Gaussian sigma range for the Hessian filter (sigma ≈ tube radius).
# MIN: half-diameter of the thinnest axon you expect to detect.
# MAX: half-diameter of the widest dendrite or proximal process.
# Too-low MIN → noise response; too-high MAX → thick structures smear.
TUBE_RADIUS_MIN_UM = 0.35
TUBE_RADIUS_MAX_UM = 3.0

# ── Number of Scales ───────────────────────────────────────────────────────────
# Log-spaced sigma values between MIN and MAX.  6–8 covers most datasets.
N_SIGMAS = 8

# ── GPU Slab Size ──────────────────────────────────────────────────────────────
# Z slices per GPU batch (core region, before overlap padding).
# Reduce by half if you get an out-of-memory error.
#   MPS (Apple Silicon) : 32–64
#   NVIDIA 8 GB         : 64–128
#   NVIDIA 24 GB        : 128–256
GPU_SLAB_SIZE = 32

# ── Frangi Hyperparameters ─────────────────────────────────────────────────────
# FRANGI_ALPHA : plate-vs-tube sensitivity (Ra = λ₂/λ₃).  Default 0.5.
# FRANGI_BETA  : blob-vs-tube sensitivity (Rb = |λ₁|/√|λ₂λ₃|).
#   β=0.5 → blobs at ~14% of peak; β=0.3 → blobs at ~0.4% (stronger suppression).
#   Lower β if somas or puncta contaminate T_combined.
FRANGI_ALPHA = 0.5
FRANGI_BETA  = 0.3

# ── Intensity Normalisation ────────────────────────────────────────────────────
# Stack is clipped to [p_low, p_high] percentiles then scaled to [0, 1].
# Clipping removes saturated pixels and dark-current offset without
# compressing the biological signal dynamic range.
NORM_PERCENTILE_LOW  = 1.0
NORM_PERCENTILE_HIGH = 99.9

print('Configuration loaded.')
print(f'  Data       : {DATA_PATH}')
print(f'  Voxel Z    : {VOXEL_Z} µm')
print(f'  Downsample : XY={DOWNSAMPLE_XY}  Z={DOWNSAMPLE_Z}')
print(f'  Tube radii : {TUBE_RADIUS_MIN_UM}–{TUBE_RADIUS_MAX_UM} µm  ({N_SIGMAS} scales)')
print(f'  GPU slab   : {GPU_SLAB_SIZE} slices')
print(f'  Frangi α/β : {FRANGI_ALPHA}/{FRANGI_BETA}')
print(f'  Norm range : [{NORM_PERCENTILE_LOW}, {NORM_PERCENTILE_HIGH}] percentile')

In [ ]:
# ── Restore from a Previous Run ───────────────────────────────────────────────
# Uncomment the block below to skip recomputation and restore a saved checkpoint.
# After restoring, visualization cells and the save cell can still be re-run.

# ck           = np.load('output/tubularity.npz')
# T_combined   = ck['T_combined']    # float32 (Z, Y, X) — tubularity map [0, 1]
# scale_idx_m  = ck['scale_idx_m']   # uint8   (Z, Y, X) — index of best sigma per voxel
# radius_map   = ck['radius_map']    # float32 (Z, Y, X) — local radius in µm
# sigmas       = ck['sigmas']        # float64 (N_SIGMAS,) — sigma values used
# voxel_iso    = float(ck['voxel_iso'])   # isotropic voxel size in µm
# orient_field = ck['orient_field']  # float16 (Z, Y, X, 3) — tube-axis eigenvectors
# stack_iso    = tifffile.imread('output/stack_iso.tif')
# print(f'Restored — T_combined={T_combined.shape}  orient_field={orient_field.shape}')

---
## Step 1 · Data Loading & Inspection

Loads the raw TIFF stack and extracts voxel size metadata.

**XY voxel size** is read from the TIFF `XResolution` tag (written by most acquisition software).
If not found, `voxel_xy` defaults to 1.0 µm and a warning is printed — set it manually.

**Z voxel size** (`VOXEL_Z`) must be set manually in Configuration because it is not reliably
stored in standard TIFF tags. Some ImageJ-saved files include it under the `spacing` key
in the ImageJ metadata block, which is printed here if found.

In [ ]:
print(f'Loading: {DATA_PATH}')
stack_raw = tifffile.imread(DATA_PATH)   # Expected axis order: (Z, Y, X)

# Handle 2D images (single Z slice) by adding a Z dimension
if stack_raw.ndim == 2:
    stack_raw = stack_raw[np.newaxis]
    print('  2D image detected — promoted to (1, Y, X)')

print(f'  Shape (Z, Y, X)  : {stack_raw.shape}')
print(f'  dtype            : {stack_raw.dtype}')
print(f'  Memory           : {stack_raw.nbytes / 1e9:.2f} GB')
print(f'  Intensity range  : {int(stack_raw.min())} – {int(stack_raw.max())}')

In [ ]:
# ── Auto-detect XY voxel size from TIFF metadata ──────────────────────────────
voxel_xy = None

with tifffile.TiffFile(DATA_PATH) as tif:
    # ImageJ / FIJI saves extra metadata in a dedicated block
    if tif.is_imagej:
        ij = tif.imagej_metadata
        print('ImageJ metadata found:')
        for k, v in ij.items():
            if k != 'labels':   # skip the per-slice label list (too long)
                print(f'  {k}: {v}')
        if 'spacing' in ij:
            print(f"\n  Z spacing in file : {ij['spacing']} {ij.get('unit', '')}")
            print(f"  → Consider setting VOXEL_Z = {ij['spacing']} in Configuration")

    # XResolution tag stores pixels-per-unit (reciprocal of µm/pixel)
    xres = tif.pages[0].tags.get('XResolution')
    if xres is not None:
        val = xres.value
        res = (val[0] / val[1]) if isinstance(val, tuple) and val[1] != 0 else float(val)
        if res > 0:
            voxel_xy = 1.0 / res   # convert pixels/µm → µm/pixel

if voxel_xy is None or voxel_xy <= 0:
    voxel_xy = 1.0
    print('WARNING: XY voxel size not found in TIFF metadata.')
    print('  Defaulting to voxel_xy = 1.0 µm. To fix, uncomment and set:')
    print('  # voxel_xy = 0.171   # µm/pixel — replace with your actual value')
else:
    print(f'\nXY voxel size (auto-detected) : {voxel_xy:.4f} µm/pixel')

print(f'Z  voxel size (config)        : {VOXEL_Z:.4f} µm/slice')
print(f'Raw anisotropy (Z/XY)         : {VOXEL_Z / voxel_xy:.2f}x')
print(f'  (After {DOWNSAMPLE_XY}x XY downsampling: {VOXEL_Z / (voxel_xy * DOWNSAMPLE_XY):.2f}x)')

---
## Step 2 · Visualization Utilities

Run this cell once. The three functions defined here are reused throughout the notebook.

| Function | Description |
|----------|-------------|
| `show_mip(vol)` | Static 3-panel max intensity projection (XY, XZ, YZ) |
| `show_slices(vol, overlay=...)` | Interactive Z-slice browser with optional coloured overlay and threshold slider |
| `compare_mip(va, vb)` | Side-by-side MIP comparison (2 rows × 3 axes) — useful for before/after checks |

All functions accept an optional `ar` argument for the Z/XY aspect ratio used in XZ and YZ panels.
Defaults to `_aniso()`, which computes the correct ratio for the **downsampled** working stack.
Pass `ar=1.0` for isotropic volumes (after Z-rescaling in Step 3c).

In [ ]:
def _aniso():
    """Z/XY aspect ratio of the downsampled working stack.
    Used to display XZ and YZ projections without distortion."""
    return (VOXEL_Z * DOWNSAMPLE_Z) / (voxel_xy * DOWNSAMPLE_XY)


def show_mip(vol, title='Max Intensity Projection', cmap='gray',
             vmin=None, vmax=None, ar=None):
    """Display three orthogonal maximum intensity projections.

    Parameters
    ----------
    vol   : ndarray (Z, Y, X)
    title : figure title
    cmap  : colormap (default 'gray')
    vmin, vmax : intensity display range (default: data min/max)
    ar    : Z/XY aspect ratio for XZ and YZ panels.
            Defaults to _aniso() for the downsampled stack.
            Pass 1.0 for isotropic stacks (e.g. stack_iso).
    """
    if ar is None:
        ar = _aniso()
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    kw = dict(cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
    axes[0].imshow(vol.max(axis=0), aspect='equal', **kw)
    axes[0].set_title('XY  (Z projection)')
    axes[1].imshow(vol.max(axis=1), aspect=ar, **kw)
    axes[1].set_title('XZ  (Y projection)')
    axes[2].imshow(vol.max(axis=2), aspect=ar, **kw)
    axes[2].set_title('YZ  (X projection)')
    for ax in axes:
        ax.axis('off')
    fig.suptitle(title, fontsize=13)
    plt.show()


def show_slices(vol, title='', cmap='gray', vmin=None, vmax=None,
                overlay=None, overlay_cmap='hot', overlay_thresh=0.1, ar=None):
    """Interactive Z-slice browser with an optional colour overlay.

    Displays three fixed panels (XY slice, XZ cross-section at y=center,
    YZ cross-section at x=center) plus an optional fourth panel showing
    a semi-transparent colour overlay on the same XY slice.

    Parameters
    ----------
    vol            : ndarray (Z, Y, X) — base grayscale volume
    overlay        : ndarray (Z, Y, X) — overlay volume (e.g. T_combined)
    overlay_thresh : float [0, 1] — fraction of overlay.max() below which
                     overlay is masked (controlled by the slider when overlay is set)
    ar             : Z/XY aspect ratio; defaults to _aniso()
    """
    if ar is None:
        ar = _aniso()
    nz  = vol.shape[0]
    out = widgets.Output()
    slider = widgets.IntSlider(
        value=nz // 2, min=0, max=nz - 1,
        description='Z slice', layout=widgets.Layout(width='600px')
    )
    tslider = widgets.FloatSlider(
        value=overlay_thresh, min=0.01, max=0.99, step=0.01,
        description='Overlay >=', layout=widgets.Layout(width='400px')
    ) if overlay is not None else None

    def _draw(z, thresh=overlay_thresh):
        with out:
            clear_output(wait=True)
            ncols = 4 if overlay is not None else 3
            fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5),
                                     constrained_layout=True)
            kw = dict(cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
            axes[0].imshow(vol[z], aspect='equal', **kw)
            axes[0].set_title(f'XY  z={z}')
            axes[1].imshow(vol[:, vol.shape[1] // 2, :], aspect=ar, **kw)
            axes[1].axhline(z, color='red', lw=0.8, alpha=0.7)
            axes[1].set_title('XZ  y=center')
            axes[2].imshow(vol[:, :, vol.shape[2] // 2], aspect=ar, **kw)
            axes[2].axhline(z, color='red', lw=0.8, alpha=0.7)
            axes[2].set_title('YZ  x=center')
            if overlay is not None:
                axes[3].imshow(vol[z], aspect='equal', **kw)
                ov = overlay[z].astype(float)
                ov_masked = np.ma.masked_where(ov < thresh * overlay.max(), ov)
                axes[3].imshow(ov_masked, cmap=overlay_cmap, alpha=0.75,
                               aspect='equal', interpolation='nearest')
                axes[3].set_title(f'Overlay  z={z}')
            for ax in axes:
                ax.axis('off')
            fig.suptitle(title, fontsize=12)
            plt.show()

    if tslider is not None:
        widgets.interactive_output(_draw, {'z': slider, 'thresh': tslider})
        display(widgets.VBox([slider, tslider]), out)
    else:
        widgets.interactive_output(_draw, {'z': slider})
        display(slider, out)
    _draw(nz // 2)


def compare_mip(va, vb, title_a='Before', title_b='After', cmap='gray',
                ar_a=None, ar_b=None):
    """Side-by-side MIP comparison: two rows (va top, vb bottom), three columns (XY, XZ, YZ).

    Parameters
    ----------
    va, vb       : ndarray (Z, Y, X) — two volumes to compare
    title_a/b    : row labels
    ar_a, ar_b   : independent Z/XY aspect ratios per row.
                   Useful when comparing pre- and post-isotropic-rescale stacks:
                   pass ar_a=_aniso() and ar_b=1.0.
    """
    if ar_a is None:
        ar_a = _aniso()
    if ar_b is None:
        ar_b = _aniso()
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
    for row, (vol, ttl, ar) in enumerate([(va, title_a, ar_a), (vb, title_b, ar_b)]):
        axes[row, 0].imshow(vol.max(axis=0), cmap=cmap, aspect='equal')
        axes[row, 0].set_title(f'{ttl} – XY')
        axes[row, 1].imshow(vol.max(axis=1), cmap=cmap, aspect=ar)
        axes[row, 1].set_title(f'{ttl} – XZ')
        axes[row, 2].imshow(vol.max(axis=2), cmap=cmap, aspect=ar)
        axes[row, 2].set_title(f'{ttl} – YZ')
    for ax in axes.ravel():
        ax.axis('off')
    plt.show()


print('Visualization utilities ready.')

In [ ]:
# Inspect the raw stack before any processing.
# Use the XZ and YZ panels to verify the anisotropy ratio looks correct.
# If the tissue appears squashed or stretched vertically, check VOXEL_Z.
show_mip(stack_raw, title='Raw Stack — Max Intensity Projection', ar=VOXEL_Z / voxel_xy)

---
## Step 3 · Preprocessing

Four sub-steps transform the raw integer stack into an isotropic float32 volume
ready for Hessian-based filtering.

### 3a · Downsampling
Stride-based subsampling along X and Y (and optionally Z).
Reduces voxel count by `DOWNSAMPLE_XY²`, making tubularity computation faster
at the cost of spatial resolution. Use `DOWNSAMPLE_XY=1` for the final production run.

### 3b · Intensity Normalization
Clips the intensity distribution to `[NORM_PERCENTILE_LOW, NORM_PERCENTILE_HIGH]` percentiles,
then scales to `[0.0, 1.0]` float32.
- **Why percentile clipping?** Raw fluorescence stacks often contain a small number of extremely
  bright pixels (saturated PMT, shot noise spikes). If we normalize to `[min, max]`, these outliers
  compress the entire biological signal into a narrow range near zero.
- **Why float32?** The Hessian filter and eigenvalue solver operate on float32 tensors on the GPU.
  Converting early avoids repeated dtype promotion.

### 3c · Isotropic Z-Rescaling
Stretches the Z axis by `aniso = vz / vxy` so that all three voxel dimensions are equal.
This is **required** for Hessian-based vesselness filters:
- The Hessian measures local curvature. If voxels are not isotropic, curvature is
  artificially different along Z vs XY, causing the filter to respond incorrectly to
  structures oriented along Z.
- The Gaussian sigma (radius) has the same physical meaning in all directions only
  after voxels are isotropic.
- `ndimage.zoom` with `order=1` (bilinear) is used. Higher orders (e.g. 3) are smoother
  but significantly slower and can introduce ringing near bright features.

> **Memory note:** Rescaling multiplies Z slice count by `aniso`. For anisotropy ~3×,
> `stack_iso` is ~3× larger than `stack_ds`. If RAM is tight, increase `DOWNSAMPLE_XY`
> rather than skipping this step.

### 3d · Sigma Conversion (µm → voxels)
Converts `TUBE_RADIUS_MIN/MAX_UM` to voxel-unit sigma values using `voxel_iso`,
the isotropic voxel size established in 3c.

In [ ]:
# 3a. Downsampling
# Stride-based subsampling — no anti-aliasing filter applied.
# For development iterations, DOWNSAMPLE_XY=2 gives a 4x speedup with acceptable quality.
stack_ds = stack_raw[::DOWNSAMPLE_Z, ::DOWNSAMPLE_XY, ::DOWNSAMPLE_XY]

# Effective voxel sizes after downsampling
vxy   = voxel_xy * DOWNSAMPLE_XY   # µm/pixel in XY
vz    = VOXEL_Z  * DOWNSAMPLE_Z    # µm/slice in Z
aniso = vz / vxy                    # Z/XY ratio — how many times coarser Z is than XY

print(f'Shape after downsampling : {stack_ds.shape}  (was {stack_raw.shape})')
print(f'XY voxel size            : {vxy:.4f} µm')
print(f'Z  voxel size            : {vz:.4f} µm')
print(f'Anisotropy (Z/XY)        : {aniso:.2f}x')
print(f'Memory                   : {stack_ds.nbytes / 1e9:.2f} GB')

In [ ]:
# 3b. Intensity normalization to float32 [0, 1]
stack_f = stack_ds.astype(np.float32)
p_low, p_high = np.percentile(stack_f, [NORM_PERCENTILE_LOW, NORM_PERCENTILE_HIGH])

print(f'Full intensity range : {stack_f.min():.0f} – {stack_f.max():.0f}')
print(f'Clipping to          : {p_low:.1f} – {p_high:.1f}')
print(f'                       ({NORM_PERCENTILE_LOW}th – {NORM_PERCENTILE_HIGH}th percentile)')

stack_norm = np.clip(stack_f, p_low, p_high)
stack_norm = (stack_norm - p_low) / (p_high - p_low)

# Explicitly cast to float32: numpy division can promote to float64.
# Keeping float32 halves memory and matches GPU tensor dtype.
stack_norm = stack_norm.astype(np.float32)

print(f'Normalized range     : {stack_norm.min():.4f} – {stack_norm.max():.4f}')
del stack_f   # free the temporary float32 copy

In [ ]:
# 3c. Isotropic Z-rescaling
# Stretch the Z axis so all voxel dimensions are equal (= vxy µm).
# Skip if already near-isotropic (aniso <= 1.2 means < 20% difference).
if aniso > 1.2:
    print(f'Rescaling Z axis by {aniso:.2f}x to achieve isotropic voxels ...')
    # order=1 (trilinear interpolation): fast and avoids ringing artifacts.
    # prefilter=False: skip the spline pre-filter used by higher orders.
    stack_iso = ndimage.zoom(stack_norm, (aniso, 1.0, 1.0), order=1, prefilter=False)
    voxel_iso = vxy
    print(f'  Shape before : {stack_norm.shape}')
    print(f'  Shape after  : {stack_iso.shape}')
    print(f'  Memory       : {stack_iso.nbytes / 1e9:.2f} GB')
else:
    stack_iso = stack_norm
    voxel_iso = vxy
    print(f'Anisotropy {aniso:.2f}x <= 1.2 — Z-rescaling skipped, voxels already near-isotropic.')

# ndimage.zoom may return float64; cast back to float32 explicitly.
stack_iso = stack_iso.astype(np.float32)

print(f'\nIsotropic voxel size : {voxel_iso:.4f} µm  (used for all downstream sigma conversions)')

In [ ]:
# 3d. Convert tube radius range from physical units (µm) to voxel-unit sigma values.
#
# Relationship between Gaussian sigma and tube radius:
#   For a bright tubular structure of radius R in an isotropic volume,
#   the Hessian eigenvalue response peaks when sigma ≈ R.
#   Therefore: sigma_voxels = tube_radius_um / voxel_iso
#
# N_SIGMAS values are spaced logarithmically (geomspace) so that each scale
# represents a constant ratio of tube size relative to the previous scale.
# This gives equal coverage per octave — the same as typical scale-space theory.
SIGMA_MIN = TUBE_RADIUS_MIN_UM / voxel_iso
SIGMA_MAX = TUBE_RADIUS_MAX_UM / voxel_iso
sigmas    = np.geomspace(SIGMA_MIN, SIGMA_MAX, N_SIGMAS).astype(np.float64)

print(f'Isotropic voxel size    : {voxel_iso:.4f} µm')
print(f'Sigma range (voxels)    : {SIGMA_MIN:.3f} – {SIGMA_MAX:.3f}\n')
print(f'{"Scale":>5}  {"σ (voxels)":>11}  {"radius (µm)":>12}  {"diameter (µm)":>14}')
print('─' * 48)
for i, s in enumerate(sigmas):
    r_um = s * voxel_iso
    print(f'{i+1:>5}  {s:>11.4f}  {r_um:>12.4f}  {2*r_um:>14.4f}')

# Slab overlap: extend each slab by OVERLAP slices on each side before filtering,
# then discard the border region after. This prevents boundary artifacts from the
# Gaussian convolution kernel being truncated at the slab edge.
# 3 × sigma_max covers the kernel radius at the largest scale (kernel decays to ~1% there).
OVERLAP = int(np.ceil(SIGMA_MAX * 3))
print(f'\nSlab overlap : {OVERLAP} voxels  (= ceil(σ_max × 3) — covers 3-sigma kernel radius at max scale)')

# Sanity check: warn if sigma_min < 0.5 voxels.
# Below 0.5 voxels the Gaussian kernel collapses to a delta function and the
# Hessian response becomes dominated by pixel noise rather than structure.
if SIGMA_MIN < 0.5:
    print(f'\nWARNING: SIGMA_MIN = {SIGMA_MIN:.3f} voxels < 0.5 — smallest scale is under-resolved.')
    print(f'  The filter at this scale will respond mostly to noise.')
    print(f'  → Raise TUBE_RADIUS_MIN_UM to at least {0.5 * voxel_iso:.3f} µm, or')
    print(f'    lower DOWNSAMPLE_XY to increase spatial resolution.')

In [ ]:
# Visual check: compare the stack before and after isotropic Z-rescaling.
# In the XZ and YZ panels the tissue should look geometrically correct after rescaling
# (e.g., spherical soma should appear round, not elongated along Z).
compare_mip(stack_norm, stack_iso,
            title_a='Before Z-rescaling (anisotropic)',
            title_b='After Z-rescaling (isotropic)',
            ar_a=aniso,   # correct aspect for the anisotropic stack
            ar_b=1.0)     # isotropic: all axes equal

---
## Step 4 · Multi-scale Tubularity & Orientation Field

### Tubularity scores

For each Gaussian scale σ the **Hessian matrix** is computed at every voxel via
separable 1D convolutions. Three eigenvalues are solved analytically (Cardano formula —
avoids `torch.linalg.eigvalsh` which is unsupported on MPS).

| Score | Filter | Formula | Best for |
|-------|--------|---------|----------|
| V_F | **Frangi** (1998) | Penalises plates (Ra) and blobs (Rb) | Dendrites, thick processes |
| V_M | **Meijering** (2004) | Modified eigenvalue mix; no blob term | Thin axons, fine varicosities |

Combined: `T_combined = √(V_F × V_M)` — geometric mean; both filters must agree.

### Orientation field

At the best-response scale per voxel, the eigenvector **v₁** (corresponding to the
eigenvalue with the smallest absolute value) gives the local **tube axis direction**.
This is computed via the cross-product null-space method — fully MPS-compatible.

`orient_field` (Z, Y, X, 3) float16 — stored in `tubularity.npz` alongside T_combined.
Used downstream for direction-consistent path selection at fiber crossings.

### Frangi formula

Sort |λ₁| ≤ |λ₂| ≤ |λ₃| (after bright-ridge sign flip):
```
Ra = λ₂/λ₃            plate ratio  (0 for tube, 1 for sheet)
Rb = |λ₁|/√|λ₂λ₃|    blob ratio   (0 for tube, large for sphere)
S  = ‖H‖_F             Frobenius norm (pre-scanned globally for γ)

V_F = (1 − e^{−Ra²/2α²}) × e^{−Rb²/2β²} × (1 − e^{−S²/2γ²})   [λ₂,λ₃ > 0 only]
```

### Meijering formula

```
νᵢ = (1−αₘ)·λᵢ + αₘ·(λ₁+λ₂+λ₃)    αₘ = 1/4 (3D)
V_M = max(νᵢ, 0),  normalised per scale to [0, 1]
```

### Global gamma pre-scan

Before the main loop, a fast pass over 25% of slabs estimates the 99th-percentile
Frobenius norm per scale. This fixes `γ` globally so it does not vary between slabs
or CPU/GPU runs — eliminating a consistency bug present in earlier versions.

### Memory layout

The full volume is processed in slabs of `GPU_SLAB_SIZE` slices, padded by `OVERLAP`
(= ceil(σ_max × 3)) on each side to prevent convolution boundary artefacts.

In [ ]:
import gc
import time

# ─────────────────────────────────────────────────────────────────────────────
# GPU Helper Functions
# Defined only when USE_GPU=True (torch imported successfully).
# These implement the same Frangi + Meijering formulas as scikit-image,
# but run natively on MPS (Apple Silicon) or CUDA (NVIDIA) GPUs.
# ─────────────────────────────────────────────────────────────────────────────

if USE_GPU:

    def _sep_conv3d(vol, kz, ky, kx):
        v = F.conv3d(vol, kz.view(1, 1, -1, 1, 1), padding=(len(kz) // 2, 0, 0))
        v = F.conv3d(v,   ky.view(1, 1, 1, -1, 1), padding=(0, len(ky) // 2, 0))
        v = F.conv3d(v,   kx.view(1, 1, 1, 1, -1), padding=(0, 0, len(kx) // 2))
        return v

    def _kernels(sigma, device, dtype=torch.float32):
        radius = max(1, int(sigma * 3.5 + 0.5))
        x  = torch.arange(-radius, radius + 1, dtype=dtype, device=device)
        g  = torch.exp(-0.5 * (x / sigma) ** 2)
        g0 = g / g.sum()
        g1 = -x / sigma ** 2 * g0
        g2 = ((x ** 2 / sigma ** 4) - 1.0 / sigma ** 2) * g0
        return g0, g1, g2

    def _hessian_components(slab_np, sigma, device, dtype=torch.float32):
        """Compute all 6 independent sigma²-normalised Hessian components on GPU."""
        g0, g1, g2 = _kernels(sigma, device, dtype)
        v = (torch.from_numpy(np.ascontiguousarray(slab_np))
             .to(device=device, dtype=dtype)
             .unsqueeze(0).unsqueeze(0))
        s   = sigma ** 2
        Dzz = _sep_conv3d(v, g2, g0, g0).squeeze() * s
        Dyy = _sep_conv3d(v, g0, g2, g0).squeeze() * s
        Dxx = _sep_conv3d(v, g0, g0, g2).squeeze() * s
        Dzy = _sep_conv3d(v, g1, g1, g0).squeeze() * s
        Dzx = _sep_conv3d(v, g1, g0, g1).squeeze() * s
        Dyx = _sep_conv3d(v, g0, g1, g1).squeeze() * s
        del v
        return Dzz, Dyy, Dxx, Dzy, Dzx, Dyx

    def _eigvalsh3x3(H):
        """Analytic eigenvalues of (N,3,3) symmetric matrices — Cardano formula.
        Used instead of torch.linalg.eigvalsh which is not supported on MPS."""
        a = H[:, 0, 0]; b = H[:, 1, 1]; c = H[:, 2, 2]
        d = H[:, 0, 1]; e = H[:, 0, 2]; f = H[:, 1, 2]
        I1 = a + b + c
        I2 = a*b + a*c + b*c - d*d - e*e - f*f
        I3 = a*(b*c - f*f) - d*(d*c - f*e) + e*(d*f - b*e)
        shift = I1 / 3.0
        p = I2 - I1 * shift
        q = -2.0 * shift**3 + I2 * shift - I3
        m       = 2.0 * torch.sqrt((-p / 3.0).clamp(min=0.0))
        cos_arg = torch.where(m > 1e-12,
                              (3.0 * q / (p * m)).clamp(-1.0, 1.0),
                              torch.zeros_like(m))
        theta = torch.acos(cos_arg) / 3.0
        pi23  = 2.0 * torch.pi / 3.0
        lam   = torch.stack([
            m * torch.cos(theta)          + shift,
            m * torch.cos(theta - pi23)   + shift,
            m * torch.cos(theta - 2*pi23) + shift,
        ], dim=1)
        return lam.sort(dim=1).values    # ascending: λ₁ ≤ λ₂ ≤ λ₃

    def _cross3(a, b):
        """Element-wise cross product of two (N, 3) tensors.
        Implemented manually to avoid torch.cross MPS compatibility issues."""
        return torch.stack([
            a[:, 1] * b[:, 2] - a[:, 2] * b[:, 1],
            a[:, 2] * b[:, 0] - a[:, 0] * b[:, 2],
            a[:, 0] * b[:, 1] - a[:, 1] * b[:, 0],
        ], dim=1)

    def _gpu_cache_clear():
        if DEVICE.type == 'mps':
            torch.mps.empty_cache()
        elif DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    def frangi_meijering_gpu(slab_np, sigma, gamma_sq,
                              frangi_alpha=0.5, frangi_beta=0.5):
        """Compute Frangi vesselness (V_F), Meijering neuriteness (V_M), and
        tube-axis orientation vector (v1) on GPU.

        Returns
        -------
        f_np  : float32 ndarray (Z, Y, X) — Frangi vesselness, unnormalized
        m_np  : float32 ndarray (Z, Y, X) — Meijering neuriteness, normalised [0,1]
        v1_np : float16 ndarray (Z, Y, X, 3) — tube axis unit vector (Hessian eigvec
                corresponding to the eigenvalue with the smallest absolute value).
                Sign convention: z-component (axis-0) >= 0.
        """
        Dzz, Dyy, Dxx, Dzy, Dzx, Dyx = _hessian_components(slab_np, sigma, DEVICE)
        Z, Y, X = Dxx.shape
        N = Z * Y * X

        # ── Assemble (N, 3, 3) Hessian matrix batch ───────────────────────────
        # Kept alive until eigenvector computation below.
        H = torch.zeros(N, 3, 3, dtype=torch.float32, device=DEVICE)
        H[:, 0, 0] = Dzz.reshape(N)
        H[:, 1, 1] = Dyy.reshape(N)
        H[:, 2, 2] = Dxx.reshape(N)
        H[:, 0, 1] = H[:, 1, 0] = Dzy.reshape(N)
        H[:, 0, 2] = H[:, 2, 0] = Dzx.reshape(N)
        H[:, 1, 2] = H[:, 2, 1] = Dyx.reshape(N)
        del Dzz, Dyy, Dxx, Dzy, Dzx, Dyx

        ev = _eigvalsh3x3(H)    # (N, 3) ascending: λ₁ ≤ λ₂ ≤ λ₃
        # NOTE: H is intentionally NOT deleted here — needed for eigenvector below.

        # Bright-ridge sign convention: negate so tube eigenvalues become positive
        ev = -ev

        # Sort by absolute value: |μ₁| ≤ |μ₂| ≤ |μ₃|
        sort_idx = ev.abs().argsort(dim=1)
        ev_s     = ev.gather(1, sort_idx)
        lam1, lam2, lam3 = ev_s[:, 0], ev_s[:, 1], ev_s[:, 2]
        del ev, sort_idx, ev_s

        eps = 1e-10

        # ── Frangi Vesselness ─────────────────────────────────────────────────
        lam2_f = lam2.clamp(min=eps)
        lam3_f = lam3.clamp(min=eps)
        Ra  = lam2_f / lam3_f
        Rb  = lam1.abs() / (lam2_f.abs() * lam3_f.abs()).sqrt()
        S2  = lam1 ** 2 + lam2 ** 2 + lam3 ** 2
        if gamma_sq is None:
            gamma_sq_use = (0.5 * S2.sqrt().max().clamp(min=eps)) ** 2
        else:
            gamma_sq_use = float(gamma_sq)
        V_F = ((1.0 - torch.exp(-Ra ** 2 / (2.0 * frangi_alpha ** 2)))
               * torch.exp(-Rb ** 2 / (2.0 * frangi_beta ** 2))
               * (1.0 - torch.exp(-S2  / (2.0 * gamma_sq_use))))
        tube_mask = ((lam2 > 0) & (lam3 > 0)).float()
        V_F = (V_F * tube_mask).clamp(min=0.0)

        # ── Meijering Neuriteness ─────────────────────────────────────────────
        alpha_m = 0.25
        sum_ev  = lam1 + lam2 + lam3
        nu      = (1.0 - alpha_m) * torch.stack([lam1, lam2, lam3], dim=1) \
                  + alpha_m * sum_ev.unsqueeze(1)
        max_idx = nu.abs().argmax(dim=1, keepdim=True)
        V_M     = nu.gather(1, max_idx).squeeze(1).clamp(min=0.0)
        max_m   = V_M.max().clamp(min=eps)
        V_M     = (V_M / max_m) * tube_mask

        f_np = V_F.reshape(Z, Y, X).contiguous().cpu().numpy()
        m_np = V_M.reshape(Z, Y, X).contiguous().cpu().numpy()

        del lam2_f, lam3_f, Ra, Rb, S2, V_F, V_M, nu, sum_ev, max_idx, tube_mask
        del lam2, lam3   # not needed for eigenvector

        # ── Tube-Axis Eigenvector (v₁) ────────────────────────────────────────
        # lam1 = -(λ₁ of H).  λ₁ is the eigenvalue with smallest |value|.
        # Eigenvector of H for λ₁:  null space of (H − λ₁·I) = (H + lam1·I).
        # Cross-product method: v₁ = rᵢ × rⱼ from rows of (H + lam1·I).
        # We pick the cross product with the largest norm for numerical stability.
        A = H.clone()
        A[:, 0, 0] += lam1
        A[:, 1, 1] += lam1
        A[:, 2, 2] += lam1
        del H, lam1

        r0, r1, r2 = A[:, 0, :], A[:, 1, :], A[:, 2, :]
        del A

        v01 = _cross3(r0, r1)
        v02 = _cross3(r0, r2)
        v12 = _cross3(r1, r2)
        del r0, r1, r2

        n01 = (v01 * v01).sum(dim=1)
        n02 = (v02 * v02).sum(dim=1)
        n12 = (v12 * v12).sum(dim=1)
        best01 = (n01 >= n02) & (n01 >= n12)
        best02 = (~best01) & (n02 >= n12)
        best12 = ~best01 & ~best02

        v1 = (best01.unsqueeze(1).float() * v01
              + best02.unsqueeze(1).float() * v02
              + best12.unsqueeze(1).float() * v12)
        del v01, v02, v12, n01, n02, n12, best01, best02, best12

        v1_norm = (v1 * v1).sum(dim=1, keepdim=True).sqrt().clamp(min=1e-10)
        v1 = v1 / v1_norm
        del v1_norm

        # Sign convention: orient so that the z-component (axis 0) is non-negative
        flip = ((v1[:, 0] < 0).float() * (-2.0) + 1.0).unsqueeze(1)
        v1 = v1 * flip
        del flip

        v1_np = v1.reshape(Z, Y, X, 3).contiguous().cpu().numpy().astype(np.float16)
        del v1
        _gpu_cache_clear()

        return f_np, m_np, v1_np

    print(f'GPU tubularity functions defined (with orientation field). Device: {DEVICE}')

else:
    print('USE_GPU=False — GPU functions skipped; CPU path will be used.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CPU Fallback Functions
# Always defined regardless of USE_GPU.
# ─────────────────────────────────────────────────────────────────────────────

from scipy.ndimage import gaussian_filter1d as _gf1

def _hessian_components_cpu(slab_np, sigma):
    """Compute 6 sigma²-normalised Hessian components on CPU.
    Matches the GPU _hessian_components convention (z=0, y=1, x=2)."""
    s2  = float(sigma) ** 2
    arr = slab_np.astype(np.float64)
    g0 = lambda a, ax: _gf1(a, sigma, axis=ax, order=0)
    g1 = lambda a, ax: _gf1(a, sigma, axis=ax, order=1)
    g2 = lambda a, ax: _gf1(a, sigma, axis=ax, order=2)
    Hzz = s2 * g0(g0(g2(arr, 0), 1), 2)   # d²/dz², smooth y, x
    Hyy = s2 * g0(g2(g0(arr, 0), 1), 2)   # smooth z, d²/dy², smooth x
    Hxx = s2 * g0(g0(g2(arr, 2), 1), 0)   # smooth z, y, d²/dx²
    Hzy = s2 * g0(g1(g1(arr, 0), 1), 2)   # d/dz, d/dy, smooth x
    Hzx = s2 * g1(g0(g1(arr, 2), 1), 0)   # d/dz, smooth y, d/dx
    Hyx = s2 * g1(g1(g0(arr, 2), 0), 1)   # smooth z, d/dy, d/dx
    return Hzz, Hyy, Hxx, Hzy, Hzx, Hyx


def _eigenvector_cpu(Hzz, Hyy, Hxx, Hzy, Hzx, Hyx):
    """Extract tube-axis eigenvector (v₁) from CPU Hessian components.
    Uses numpy.linalg.eigh — correct but ~10× slower than GPU cross-product path.

    Returns
    -------
    v1_np : float16 ndarray (Z, Y, X, 3) — tube axis unit vectors
    """
    Z, Y, X = Hzz.shape
    N = Z * Y * X
    H = np.zeros((N, 3, 3), dtype=np.float32)
    H[:, 0, 0] = Hzz.reshape(N)
    H[:, 1, 1] = Hyy.reshape(N)
    H[:, 2, 2] = Hxx.reshape(N)
    H[:, 0, 1] = H[:, 1, 0] = Hzy.reshape(N)
    H[:, 0, 2] = H[:, 2, 0] = Hzx.reshape(N)
    H[:, 1, 2] = H[:, 2, 1] = Hyx.reshape(N)

    # eigh returns (N,3) eigenvalues ascending + (N,3,3) eigenvectors as columns
    eigenvalues, eigenvectors = np.linalg.eigh(H.astype(np.float64))

    # Select eigenvector for the eigenvalue with smallest absolute value (tube axis)
    abs_order = np.abs(eigenvalues).argsort(axis=1)   # (N, 3)
    idx_v1    = abs_order[:, 0]                        # (N,)
    v1 = eigenvectors[np.arange(N), :, idx_v1].astype(np.float32)  # (N, 3)

    # Sign convention: non-negative z-component
    sign = np.where(v1[:, 0] >= 0, np.float32(1), np.float32(-1))
    v1 = v1 * sign[:, None]

    return v1.reshape(Z, Y, X, 3).astype(np.float16)


def frangi_meijering_cpu(slab_np, sigma, gamma_sq=None,
                          frangi_alpha=0.5, frangi_beta=0.5):
    """CPU path: scikit-image Frangi + Meijering, plus CPU eigenvector extraction.

    Returns
    -------
    f_np  : float32 (Z, Y, X)    — Frangi vesselness
    m_np  : float32 (Z, Y, X)    — Meijering neuriteness
    v1_np : float16 (Z, Y, X, 3) — tube axis orientation vector
    """
    sf = frangi(slab_np, sigmas=[sigma], black_ridges=False,
                alpha=frangi_alpha, beta=frangi_beta)
    sm = meijering(slab_np, sigmas=[sigma], black_ridges=False)

    Hzz, Hyy, Hxx, Hzy, Hzx, Hyx = _hessian_components_cpu(slab_np, sigma)
    v1_np = _eigenvector_cpu(Hzz, Hyy, Hxx, Hzy, Hzx, Hyx)

    return sf.astype(np.float32), sm.astype(np.float32), v1_np


def compute_tubularity(slab_np, sigma, gamma_sq,
                        frangi_alpha=0.5, frangi_beta=0.5):
    """Unified dispatch: GPU if available, else CPU.
    Returns (f_np, m_np, v1_np)."""
    if USE_GPU:
        return frangi_meijering_gpu(slab_np, sigma, gamma_sq,
                                    frangi_alpha=frangi_alpha,
                                    frangi_beta=frangi_beta)
    else:
        return frangi_meijering_cpu(slab_np, sigma, gamma_sq=None,
                                    frangi_alpha=frangi_alpha,
                                    frangi_beta=frangi_beta)


print('CPU fallback functions defined (frangi + meijering + eigenvector).')
print('compute_tubularity() returns (f_np, m_np, v1_np).')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Global Gamma Pre-scan
#
# Purpose: compute gamma_per_sigma[i] = (S_p99[i] / 2)² for each sigma scale,
# where S_p99[i] is the 99th-percentile Frobenius norm of the Hessian
# across the entire volume at scale i. Using the 99th percentile instead of
#
# Strategy: sample every PRESCAN_STRIDE-th slab (default every 4th = 25% coverage).
# This gives a good estimate of the global maximum without processing the full volume.
# If your data has strong regional intensity variation, increase coverage by lowering
# PRESCAN_STRIDE (e.g. 2 = 50% coverage, 1 = 100% = full pass).
#
# Why 99th percentile instead of max?
# max(S) is dominated by a single bright outlier (e.g. a soma or fluorescent
# artefact), inflating gamma and suppressing most tubular structures — only
# the few brightest voxels survive the background-suppression term.
# The 99th percentile ignores the top 1% of S values, giving a gamma that
# is representative of real structures while rejecting bright point outliers.
#
# Frobenius norm computed from Hessian components directly (avoids eigenvalue solver):
#   S² = Dzz² + Dyy² + Dxx² + 2·(Dzy² + Dzx² + Dyx²)
# The factor of 2 on off-diagonal terms is essential: for a symmetric matrix H,
#   ‖H‖²_F = tr(H^T H) = Σᵢ λᵢ² = Dzz² + Dyy² + Dxx² + 2Dzy² + 2Dzx² + 2Dyx²
#
# Note: the CPU path (USE_GPU=False) skips this pre-scan because scikit-image's
# frangi() handles gamma internally. gamma_per_sigma is filled with NaN in that case.
# ─────────────────────────────────────────────────────────────────────────────

PRESCAN_STRIDE = 4   # sample every Nth slab; 4 = 25% coverage
nz      = stack_iso.shape[0]
n_slabs = int(np.ceil(nz / GPU_SLAB_SIZE))
n_sampled = n_slabs // PRESCAN_STRIDE + 1

gamma_per_sigma = np.zeros(len(sigmas), dtype=np.float64)

if USE_GPU:
    print(f'Pre-scan: computing global S_p99 per scale on {DEVICE}')
    print(f'  Sampling {n_sampled} of {n_slabs} slabs (every {PRESCAN_STRIDE}th = '
          f'{100//PRESCAN_STRIDE}% coverage)\n')
    t_pre = time.time()

    for i, sigma in enumerate(sigmas):
        s_max_i = 0.0

        for slab_i, z0 in enumerate(range(0, nz, GPU_SLAB_SIZE)):
            if slab_i % PRESCAN_STRIDE != 0:
                continue   # skip non-sampled slabs

            z1  = min(z0 + GPU_SLAB_SIZE, nz)
            z0p = max(0, z0 - OVERLAP)      # padded start (for kernel boundary)
            z1p = min(nz, z1 + OVERLAP)     # padded end
            slab = stack_iso[z0p:z1p]       # numpy view — no data copy

            Dzz, Dyy, Dxx, Dzy, Dzx, Dyx = _hessian_components(slab, sigma, DEVICE)

            # Frobenius norm²: factor of 2 on off-diagonal (symmetric matrix identity)
            S2 = (Dzz**2 + Dyy**2 + Dxx**2
                  + 2.0 * (Dzy**2 + Dzx**2 + Dyx**2))
            del Dzz, Dyy, Dxx, Dzy, Dzx, Dyx

            # 99th-percentile of S = sqrt(S²) across this slab.
            # Convert to numpy for percentile — avoids both MPS quantile
            # incompatibility and torch.quantile tensor-size limit.
            # numpy percentile avoids torch.quantile tensor-size limit
            s_p99 = float(np.percentile(S2.sqrt().reshape(-1).cpu().numpy(), 99))
            s_max_i = max(s_max_i, s_p99)
            del S2
            _gpu_cache_clear()

        gamma_per_sigma[i] = (s_max_i / 2.0) ** 2
        print(f'  scale {i+1}/{len(sigmas)}  sigma={sigma:.4f} vox  '
              f'S_p99={s_max_i:.5f}  gamma={np.sqrt(gamma_per_sigma[i]):.5f}')

    print(f'\nPre-scan complete in {time.time() - t_pre:.1f}s')
    print()

    # Sanity check: gamma should increase monotonically with sigma
    # (larger structures → larger Hessian magnitudes at the matching scale).
    # A non-monotonic gamma may indicate the sampling is too sparse.
    if not all(gamma_per_sigma[i] <= gamma_per_sigma[i+1]
               for i in range(len(gamma_per_sigma)-1)):
        print('NOTE: gamma values are not strictly monotonic across scales.')
        print('  This can happen with sparse sampling (high PRESCAN_STRIDE) on')
        print('  inhomogeneous data. Consider lowering PRESCAN_STRIDE if concerned.')
    else:
        print('Gamma values increase monotonically with scale — pre-scan looks good.')

else:
    gamma_per_sigma[:] = np.nan
    print('USE_GPU=False — pre-scan skipped (scikit-image handles gamma internally).')
    print('gamma_per_sigma = NaN; compute_tubularity() will use scikit-image defaults.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Main Tubularity Loop
#
# For each slab:
#   1. Load padded slab (core + OVERLAP on each side)
#   2. For each sigma: run compute_tubularity() → (V_F, V_M, v1)
#   3. Trim overlap padding → extract core region
#   4. Update per-slab running-max accumulators (T_f_slab, T_m_slab)
#   5. Record which sigma index gave the highest Meijering response (scale_idx_m)
#   6. Update orientation field (orient_field) at best-scale voxels
#   7. Write geometric mean of max-Frangi and max-Meijering to T_combined
#   8. Track global maxima for final normalisation
#
# After all slabs: normalise T_combined globally (in-place).
# ─────────────────────────────────────────────────────────────────────────────

for _v in ('stack_ds', 'stack_f', 'stack_norm'):
    if _v in globals():
        del globals()[_v]
gc.collect()

nz      = stack_iso.shape[0]
n_slabs = int(np.ceil(nz / GPU_SLAB_SIZE))
_Y, _X  = stack_iso.shape[1], stack_iso.shape[2]

# ── Output arrays ─────────────────────────────────────────────────────────────
T_combined   = np.zeros(stack_iso.shape, dtype=np.float32)
scale_idx_m  = np.zeros(stack_iso.shape, dtype=np.uint8)
# orient_field: tube-axis eigenvector at the best-Meijering scale per voxel.
# float16 halves memory vs float32; angular precision loss < 0.5° for unit vectors.
orient_field = np.zeros((*stack_iso.shape, 3), dtype=np.float16)

max_f_global = 0.0
max_m_global = 0.0

print(f'Stack shape  : {stack_iso.shape}')
print(f'Slab size    : {GPU_SLAB_SIZE}  |  Overlap : {OVERLAP}  |  Slabs : {n_slabs}')
print(f'Sigmas       : {np.round(sigmas, 4)}')
print(f'Device       : {DEVICE if USE_GPU else "CPU (scikit-image)"}')
print(f'Memory       : T_combined {T_combined.nbytes/1e6:.0f} MB'
      f' + scale_idx_m {scale_idx_m.nbytes/1e6:.0f} MB'
      f' + orient_field {orient_field.nbytes/1e6:.0f} MB\n')

t0 = time.time()

for slab_i, z0 in enumerate(range(0, nz, GPU_SLAB_SIZE)):
    z1   = min(z0 + GPU_SLAB_SIZE, nz)
    z0p  = max(0, z0 - OVERLAP)
    z1p  = min(nz, z1 + OVERLAP)
    core = slice(z0 - z0p, z1 - z0p)
    sz   = z1 - z0

    T_f_slab    = np.zeros((sz, _Y, _X), dtype=np.float32)
    T_m_slab    = np.zeros_like(T_f_slab)
    orient_slab = np.zeros((sz, _Y, _X, 3), dtype=np.float16)

    slab = stack_iso[z0p:z1p]

    print(f'  slab {slab_i+1:2d}/{n_slabs}  z={z0}–{z1}', end='  scales:', flush=True)

    for i, sigma in enumerate(sigmas):
        gamma_sq = gamma_per_sigma[i] if USE_GPU else None

        sf, sm, v1 = compute_tubularity(slab, sigma, gamma_sq,
                                         frangi_alpha=FRANGI_ALPHA,
                                         frangi_beta=FRANGI_BETA)

        sf_c = sf[core]
        sm_c = sm[core]
        v1_c = v1[core]   # (sz, Y, X, 3) — trim overlap padding

        upd_f = sf_c > T_f_slab
        T_f_slab[upd_f] = sf_c[upd_f]

        upd_m = sm_c > T_m_slab
        T_m_slab[upd_m]            = sm_c[upd_m]
        scale_idx_m[z0:z1][upd_m] = i
        orient_slab[upd_m]         = v1_c[upd_m]   # keep eigvec at winning scale

        del sf, sm, v1, sf_c, sm_c, v1_c, upd_f, upd_m
        gc.collect()
        print(f' {i+1}', end='', flush=True)

    max_f_global = max(max_f_global, float(T_f_slab.max()))
    max_m_global = max(max_m_global, float(T_m_slab.max()))

    T_combined[z0:z1]   = np.sqrt(T_f_slab * T_m_slab)
    orient_field[z0:z1] = orient_slab   # write completed slab into full volume

    del T_f_slab, T_m_slab, orient_slab
    gc.collect()
    print(f'  [{time.time()-t0:.0f}s]')

# ── Global normalisation ───────────────────────────────────────────────────────
norm_factor = np.sqrt(max_f_global * max_m_global) + 1e-10
T_combined /= norm_factor

# ── Radius map ────────────────────────────────────────────────────────────────
radius_map = (sigmas[scale_idx_m] * voxel_iso).astype(np.float32)

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s  ({elapsed/60:.1f} min)')
print(f'T_combined   : {T_combined.min():.4f} – {T_combined.max():.4f}')
print(f'radius_map   : {radius_map.min():.4f} – {radius_map.max():.4f} µm')
print(f'orient_field : shape {orient_field.shape}  dtype {orient_field.dtype}')
print(f'norm_factor  : {norm_factor:.6f}')

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# ── Save checkpoint ───────────────────────────────────────────────────────────
np.savez_compressed(
    'output/tubularity.npz',
    T_combined   = T_combined,              # float32  (Z, Y, X)     — tubularity map [0,1]
    scale_idx_m  = scale_idx_m,             # uint8    (Z, Y, X)     — best sigma index per voxel
    radius_map   = radius_map,              # float32  (Z, Y, X)     — local tube radius (µm)
    sigmas       = sigmas,                  # float64  (N_SIGMAS,)   — sigma values used
    voxel_iso    = np.float32(voxel_iso),   # float32  scalar        — isotropic voxel size (µm)
    orient_field = orient_field,            # float16  (Z, Y, X, 3) — tube-axis eigenvectors
)
tifffile.imwrite('output/stack_iso.tif',  stack_iso,  photometric='minisblack')
tifffile.imwrite('output/T_combined.tif', T_combined, photometric='minisblack')

print('Saved → output/tubularity.npz  (T_combined, scale_idx_m, radius_map, orient_field, ...)')
print('Saved → output/stack_iso.tif')
print('Saved → output/T_combined.tif')
print()
print('For step2, set:')
print("  TUBULARITY_CHECKPOINT = 'output/tubularity.npz'")
print("  STACK_ISO_PATH        = 'output/stack_iso.tif'")

# ── Sanity checks ─────────────────────────────────────────────────────────────
dominant_sigma = sigmas[np.bincount(scale_idx_m.ravel()).argmax()]

print(f'\nSanity checks:')
print(f'  Most common winning scale : sigma={dominant_sigma:.4f} vox'
      f'  → radius={dominant_sigma * voxel_iso:.4f} µm')

# Check orient_field unit vectors
v_norms = np.linalg.norm(orient_field.astype(np.float32), axis=-1)
nonzero  = v_norms > 0.1
if nonzero.any():
    mean_norm = float(v_norms[nonzero].mean())
    print(f'  orient_field mean |v| at non-zero voxels : {mean_norm:.4f}  (should be ≈1.0)')

assert T_combined.dtype == np.float32,   'dtype error'
assert T_combined.max() <= 1.001,        'normalisation error'
assert not np.any(np.isnan(T_combined)), 'NaN in T_combined'
assert not np.any(np.isinf(T_combined)), 'Inf in T_combined'
assert scale_idx_m.max() < N_SIGMAS,     'scale_idx_m out of range'
assert orient_field.shape == (*T_combined.shape, 3), 'orient_field shape mismatch'
print('  All assertions passed.')

In [ ]:
# Visualize the two key outputs:
#   Left  : T_combined XY max-projection — shows where tubular structures were detected
#   Right : radius_map XY max-projection — shows estimated local tube radius in µm
#           (warm colors = thin structures; cool colors = thick structures)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

im0 = axes[0].imshow(T_combined.max(axis=0), cmap='hot', aspect='equal')
axes[0].set_title('Combined tubularity (T_combined) — XY MIP')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(radius_map.max(axis=0), cmap='viridis', aspect='equal',
                     vmin=TUBE_RADIUS_MIN_UM, vmax=TUBE_RADIUS_MAX_UM)
axes[1].set_title('Local tube radius (µm) — XY MIP')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], label='radius (µm)')

plt.suptitle(
    f'v2 result  |  {N_SIGMAS} scales  |  '
    f'{TUBE_RADIUS_MIN_UM}–{TUBE_RADIUS_MAX_UM} µm  |  device={DEVICE}',
    fontsize=11
)
plt.show()

In [ ]:
# Interactive slice browser with tubularity overlay.
# Use the 'Overlay >=' slider to adjust the display threshold:
#   - Lower threshold: more structures shown (may include noise)
#   - Higher threshold: only the strongest tubularity responses
# This slider only affects the visualisation, not any saved parameter.
show_slices(stack_iso,
            title='Tubularity v2 — Slice Browser with Overlay',
            overlay=T_combined,
            overlay_thresh=0.10)

---
## Roadmap

| Step | Status | Description |
|------|--------|-------------|
| 1 | ✅ | Load TIFF; auto-detect XY voxel size |
| 2 | ✅ | Visualisation utilities |
| 3 | ✅ | Downsample → normalise → isotropic rescale → sigma conversion |
| 4 | ✅ | Multi-scale Frangi + Meijering tubularity + orientation field |

**Next:** `step2.ipynb`

```python
TUBULARITY_CHECKPOINT = 'output/tubularity.npz'
STACK_ISO_PATH        = 'output/stack_iso.tif'
```